# Week 1: Are they as good as their record?

**Series:** Real or Lucky? (NFL) · **Concept:** luck vs skill

Set `TEAM` below, then run **Restart Kernel and Run All Cells**, or from the repo root:

```powershell
analysis/.venv/Scripts/python analysis/tools/run_notebook.py analysis/week01-real-or-lucky.ipynb --param TEAM=MIN
```

Before committing: clear all outputs and run `analysis/tools/check_notebooks.py`.

In [ ]:
TEAM = "MIN"
SEASON = None  # None means the latest season with completed games
THROUGH_WEEK = None  # None means the latest week whose games are all final

In [ ]:
import sys
from dataclasses import asdict
from pathlib import Path

# Find analysis/ whether this notebook runs there, in a subfolder, or from the repo root.
here = Path.cwd().resolve()
ANALYSIS = next(d for d in (here, *here.parents, here / "analysis") if (d / "lib" / "style.py").is_file())
sys.path.insert(0, str(ANALYSIS))

import pandas as pd

from lib import nfl_queries, style
from lib import nfl_team_report as report
from lib import real_or_lucky_posts as posts_lib
from lib.db import read_engine

engine = read_engine()
style.apply_style()

## 1. The question

The team has a great record (or a bad one). Is it as good as that record says, or has the record run ahead of how
well it has actually played?

## 2. The concept: luck vs skill

A record tells you who won. Points scored and allowed tell you how well a team has played, and over a season they
track wins closely. **Pythagorean wins** turn a team's points into the record they usually produce. When a record
runs well ahead of its points, close wins and turnovers are usually part of the reason, and those tend not to hold
up as well as the points do.

## 3. Data

- **This season:** regular-season scores, per-play efficiency and turnovers from the Lineform database (read only),
  through the last week whose games are all final.
- **Turnover history:** turnovers from 2024 on (when they are first stored), to measure how well early turnover
  margin tracks the rest of a season.

In [ ]:
HISTORY_FIRST_SEASON = 2012                           # first season with stored NFL scores
games = nfl_queries.games(engine, HISTORY_FIRST_SEASON, 2100)
season = SEASON or int(games.loc[games["status"] == "completed", "season"].max())
through = THROUGH_WEEK or report.last_final_week(games, season)
carryover_seasons = list(range(2024, season))         # player turnovers are stored from 2024

efficiency = nfl_queries.efficiency(engine, season, season)
giveaways = nfl_queries.giveaways(engine, min(carryover_seasons), season)

this_season = report.played_between(games, season, 1, through)
print(f"{TEAM} {season}, data through week {through}")
print(f"This season: {this_season['game_id'].nunique()} games played; turnover history: "
      f"{carryover_seasons[0]}-{carryover_seasons[-1]}")

## 4. Method

1. **Record vs points:** Pythagorean wins are points scored to the 2.37 power, over that plus points allowed to the
   same power, times games played. Rounded to whole games (halves up), that is the record a team's points usually
   produce.
2. **Luck label:** a record at least 0.75 wins above its Pythagorean wins is lucky, 0.75 below is unlucky, anything
   between is about right (the same rule as Save the Season).
3. **Where the gap comes from:** one-score games (8 points or fewer), turnover margin, and offense and defense ranks
   by EPA per play (*expected points added: how much each play helps the score*) and success rate.
4. **Unbeaten teams:** a perfect record almost always runs ahead of its points, so an unbeaten team is compared with
   every team since 2012 that won its first games too (same number of games, regular season, finished seasons
   only): how many of them had points saying they should have lost once, how many had better points, and how they
   finished.
5. **Verdict, fixed before looking at any team:**

| Luck label | Point differential | Verdict |
|---|---|---|
| about right | any | Real |
| unlucky | any | Better than their record |
| lucky | positive | Mostly real, some luck |
| lucky | zero or negative | Lucky |

A lucky verdict is not a prediction that the team will lose; it says the record has run ahead of the points.

## 5. Analysis

In [ ]:
carryover = report.turnover_carryover(giveaways, games, carryover_seasons, through)
print(f"Turnover margin through week {through} vs the rest of the season: "
      f"r = {carryover.correlation:.2f} ({carryover.team_seasons} team-seasons, a small sample)")

In [ ]:
snapshot = report.season_snapshot(TEAM, season, through, games=games, efficiency=efficiency,
                                   giveaways=giveaways)
verdict = report.real_verdict(snapshot)
points = report.points_record(snapshot.pythagorean_wins, snapshot.games)
pd.Series({**asdict(snapshot), "points_record": points, "verdict": verdict},
          name=report.team_name(TEAM)).to_frame()

In [ ]:
league = report.standings(games, season, through)
league = league.assign(actual_wins=league["wins"] + league["ties"] / 2)
league = league.assign(luck_gap=league["actual_wins"] - league["pythagorean_wins"])
league.sort_values("luck_gap", ascending=False)[["wins", "losses", "ties", "point_diff", "pythagorean_wins",
                                                  "luck_gap"]].round(2).head(10)

In [ ]:
unbeaten = snapshot.losses == 0 and snapshot.ties == 0
last_unbeaten = unbeaten and report.is_last_unbeaten(league, TEAM)
history = (report.unbeaten_history(games, list(range(HISTORY_FIRST_SEASON, season)), snapshot.games,
                                   snapshot.pythagorean_wins, points_for=snapshot.points_for,
                                   points_against=snapshot.points_against) if unbeaten else None)
print(f"Last unbeaten team: {last_unbeaten}")

SCORING_LAST_N = 5                                    # the follow-up clip's "last 5 games"
scoring = nfl_queries.scoring(engine, season, season)
scoring_run = report.scoring_run(scoring, games, TEAM, season, through, SCORING_LAST_N)
research = posts_lib.Research(
    adjustment=report.opponent_adjustment(games, TEAM, season, through),
    opponents_record=report.opponents_record(games, TEAM, season, through),
    field_goals=report.field_goal_rank(scoring, games, TEAM, season, through),
)
print(scoring_run)
print(research)
history

## 6. The answer

Printed below from the numbers, then the main caveat.

In [ ]:
print(posts_lib.opening_sentence(snapshot))
print(f"Verdict: {verdict}. {posts_lib.VERDICT_SENTENCES[verdict]}")
print(f"Caveat: {snapshot.games} games is a small sample.")

## 7. Chart export

One point: record vs points. Every team is a dot; the dashed line is where a record matches its points. Above the
line, a team has won more than its points usually produce.

In [ ]:
CHART_NAME = f"week01-real-or-lucky-{TEAM.lower()}"
nickname = report.nickname(TEAM)
title = posts_lib.chart_title(snapshot)
subtitle = f"Week {through} · above the line = record better than points"
source = f"Source: nflverse · data through week {through}, {season}"
top = int(league["games"].max())
others = league.drop(index=TEAM)
team = league.loc[TEAM]

for size in style.SIZES:
    fig, ax = style.new_figure(size)
    ax.plot([0, top], [0, top], color=style.TEXT_2, linewidth=2, linestyle="--", zorder=1)
    ax.scatter(others["pythagorean_wins"], others["actual_wins"], s=260, color=style.GRAY, zorder=2)
    ax.scatter([team["pythagorean_wins"]], [team["actual_wins"]], s=520, color=style.ACCENT, zorder=3)
    # Above the dot: teams with the same win total sit on either side of it.
    ax.annotate(nickname, (team["pythagorean_wins"], team["actual_wins"]), xytext=(0, 20),
                textcoords="offset points", ha="center", va="bottom", color=style.ACCENT,
                fontproperties=style.font("body_bold"), fontsize=style.px(34))
    ax.set_xlim(-0.3, top + 0.3)
    ax.set_ylim(-0.3, top + 0.3)
    ax.set_xticks(range(top + 1))
    ax.set_yticks(range(top + 1))
    ax.set_xlabel("Wins their points usually produce")
    ax.set_ylabel("Actual wins")
    style.finish(fig, ax, title=title, subtitle=subtitle, source=source)
    print(style.export(fig, CHART_NAME, size))

### Thumbnail

Headline: 3 to 5 words. No team logos or player photos. The config is saved in `thumbnails/configs/` so the
thumbnail can be rebuilt later.

In [ ]:
import json

from thumbnails.config import ANALYSIS_DIR, load_config
from thumbnails.make_thumbnail import render

thumbnail = {
    "headline": f"Are the {nickname} for real?",
    "subline": f"{posts_lib.record(snapshot)} through week {through}",
    "chart": f"{CHART_NAME}-feed.png",
    "sport": "nfl",
    "callout": {"value": points, "label": "record their points usually produce"},
    "name": CHART_NAME,
}
config_path = ANALYSIS_DIR / "thumbnails" / "configs" / f"{CHART_NAME}.json"
config_path.write_text(json.dumps(thumbnail, indent=2) + "\n", encoding="utf-8")
render(load_config(config_path))

## 8. Draft posts

Generated from the numbers above into `output/week01-real-or-lucky-<team>-posts.md`. Edit them before posting. No
betting advice, no picks, no claims the data does not support.

In [ ]:
posts = posts_lib.build_posts(snapshot, turnover_carryover=carryover.correlation,
                              luck_gaps=league["luck_gap"].to_dict(), history=history, last_unbeaten=last_unbeaten,
                              scoring=scoring_run, research=research)
posts_path = style.OUTPUT_DIR / f"{CHART_NAME}-posts.md"
posts_path.write_text(posts_lib.to_markdown(snapshot, posts), encoding="utf-8")
print(posts_path)
print(posts_lib.to_markdown(snapshot, posts))

### Video data

Every number the `RealOrLucky` video shows, written to `output/week01-real-or-lucky-<team>-video.json`. Render it from
`analysis/video/`:

```powershell
npx remotion render src\index.ts RealOrLucky out\week01-real-or-lucky-min.mp4 --props=..\output\week01-real-or-lucky-min-video.json
```

In [ ]:
import json

from lib import real_or_lucky_video

video_data = real_or_lucky_video.video_data(snapshot, games, turnover_carryover=carryover.correlation,
                                            history=history, last_unbeaten=last_unbeaten,
                                            adjustment=research.adjustment, scoring=scoring_run)
video_path = style.OUTPUT_DIR / f"{CHART_NAME}-video.json"
video_path.write_text(json.dumps(video_data, indent=2) + "\n", encoding="utf-8")
print(video_path)
video_data

## 9. What I learned

- **About the concept:** a great record and a great team are related but not the same thing; close games are where
  they drift apart.
- **About the method:** Pythagorean wins need only points scored and allowed, so the check works for any team after
  a few weeks, but a handful of games is still a small sample.
- **Something that surprised me:** check the follow-up clip draft; it names the most surprising number for this team.

**New glossary terms** (added to `lib/glossary.md`): points record.